# Frontend Pipeline

实现 notebook 里定义的最小前端全流程，并最终生成 `macro_op_list`。


## 流程
- 初始化 `NandConfig`、`InferenceConfig`、`Qwen3ModelConfig`
- 初始化 `Qwen3DecoderLayer`
- 使用 `NxTracer` trace 得到计算图
- 运行 `NormalizePass` 获得简化图
- 用 `build_kv_cache_state` 构造 `KVCacheState`
- 构造 `NxGraphMeta` 并写入 `graph.meta`
- 运行 `CodeGenPass` 生成 `macro_op_list`
- 断言关键输出存在


In [1]:
import json
from collections import Counter
from pathlib import Path

import torch
from torch.fx import GraphModule

from nandmachine.commands.macro import FlashAttnOp, MatMulOp, VectorOp
from nandmachine.config.config import NandConfig
from nandmachine.config.inference_config import InferenceConfig, ParallelConfig
from nandmachine.config.model_config import Qwen3ModelConfig
from nandmachine.frontend.core.graph.base import NxGraphMeta, NxTracer
from nandmachine.frontend.core.passes.cod_gen import CodeGenPass
from nandmachine.frontend.core.passes.normalize import NormalizePass
from nandmachine.frontend.network.qwen3 import Qwen3DecoderLayer
from nandmachine.frontend.utlis import build_kv_cache_state

MODEL_CARD_PATH = Path("model_cards/qwen3-8B.json")
assert MODEL_CARD_PATH.exists(), MODEL_CARD_PATH


def node_summary(graph_module: GraphModule) -> list[tuple[str, str]]:
    return [(node.op, str(node.target)) for node in graph_module.graph.nodes]


def macro_summary(macro_op_list: list[object]) -> Counter:
    return Counter(type(op).__name__ for op in macro_op_list)


In [2]:
import logging

logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [3]:
model_config = Qwen3ModelConfig.from_dict(json.loads(MODEL_CARD_PATH.read_text()))
nand_config = NandConfig(
    num_channels=1,
    num_plane=1,
    num_block=4,
    num_pages=16,
    tRead=1.0,
    tWrite=2.0,
    tErase=3.0,
    page_size=4,
    sram_threshold=1024,
)
inference_config = InferenceConfig(
    batch_size=2,
    input_sequence_length=8,
    output_sequence_length=4,
    weight_bits=16,
    activation_bits=16,
    kv_cache_bits=16,
    kv_block_size_bytes=1024,
    parallel_config=ParallelConfig(num_ranks=1),
)
kv_cache_state = build_kv_cache_state(nand_config, model_config, inference_config)
graph_meta = NxGraphMeta(
    nand_config=nand_config,
    model_config=model_config,
    inference_config=inference_config,
    kv_cache_state=kv_cache_state,
)

print(model_config)
print(kv_cache_state)


Qwen3ModelConfig(attention_type='gqa', hidden_size=4096, num_attention_heads=32, num_key_value_heads=8, max_position_embeddings=40960, intermediate_size=12288, hidden_act='silu', head_dim=128, rms_norm_eps=1e-06, attention_bias=False, rope_theta=1000000, num_hidden_layers=36)
KVCacheState(total_kv_cache_size_per_layer=98304, num_nand_pages_per_layer=24, num_hyper_pages_per_layer=24, kv_block_size_tokens=1, num_kv_blocks=96, kv_cache_num_pages_per_layer=24)


In [5]:
with torch.device("meta"):
    model = Qwen3DecoderLayer(model_config,tp_size=1)
    graph = NxTracer().trace(model)
    graph_module = GraphModule(model, graph)

NormalizePass().transform(graph_module)
graph_module.graph.meta = {CodeGenPass.GRAPH_META_KEY: graph_meta}
CodeGenPass().transform(graph_module)

macro_op_list = graph_module.graph.meta[CodeGenPass.MACRO_OP_LIST_META_KEY]
vector_ops = [op for op in macro_op_list if isinstance(op, VectorOp)]

print("normalized nodes:", node_summary(graph_module))
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("vector op summary:", [(op.vector_op_type, op.vector_shape) for op in vector_ops])


2026-03-27 19:52:33,121 - INFO - Processed node=input_layernorm generated_macro_ops=1
2026-03-27 19:52:33,122 - INFO - Processed node=self_attn_qkv_proj generated_macro_ops=144
2026-03-27 19:52:33,123 - INFO - Processed node=self_attn_q_norm generated_macro_ops=1
2026-03-27 19:52:33,124 - INFO - Processed node=self_attn_k_norm generated_macro_ops=1
2026-03-27 19:52:33,124 - INFO - Processed node=self_attn_rotary_emb generated_macro_ops=0
2026-03-27 19:52:33,126 - INFO - Processed node=self_attn_attn generated_macro_ops=72
2026-03-27 19:52:33,126 - INFO - Processed node=self_attn_o_proj generated_macro_ops=96


2026-03-27 19:52:33,127 - INFO - Processed node=post_attention_layernorm generated_macro_ops=1
2026-03-27 19:52:33,128 - INFO - Processed node=mlp_gate_up_proj generated_macro_ops=576
2026-03-27 19:52:33,128 - INFO - Processed node=mlp_act_fn generated_macro_ops=1
2026-03-27 19:52:33,129 - INFO - Processed node=mlp_down_proj generated_macro_ops=294


normalized nodes: [('placeholder', 'hidden_states'), ('call_module', 'input_layernorm'), ('call_module', 'self_attn.qkv_proj'), ('call_module', 'self_attn.q_norm'), ('call_module', 'self_attn.k_norm'), ('get_attr', 'self_attn._dummy_positions'), ('call_module', 'self_attn.rotary_emb'), ('call_module', 'self_attn.attn'), ('call_module', 'self_attn.o_proj'), ('call_function', '<built-in function add>'), ('call_module', 'post_attention_layernorm'), ('call_module', 'mlp.gate_up_proj'), ('call_module', 'mlp.act_fn'), ('call_module', 'mlp.down_proj'), ('call_function', '<built-in function add>')]
macro op count: 1187
macro op type summary: {'VectorOp': 5, 'SramPrefetch': 394, 'MatMulOp': 370, 'SramPrefetchRelease': 394, 'FlashAttnOp': 24}
vector op summary: [('rms_norm', [2, 4096]), ('rms_norm', [2, 128]), ('rms_norm', [2, 128]), ('rms_norm', [2, 4096]), ('silu_mul', [2, 12288])]


In [6]:
assert macro_op_list
assert any(isinstance(op, MatMulOp) for op in macro_op_list)
assert any(isinstance(op, FlashAttnOp) for op in macro_op_list)
assert any(op.vector_op_type == "rms_norm" for op in vector_ops)
assert any(op.vector_op_type == "silu_mul" for op in vector_ops)
assert all(all(dim > 0 for dim in op.vector_shape) for op in vector_ops)

print("frontend pipeline completed successfully")


frontend pipeline completed successfully


In [7]:
print(macro_op_list)

[VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 4096]), SramPrefetch(id=2, input_ops=[], num_prefetch_pages=256), MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16), SramPrefetchRelease(id=4, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)]), SramPrefetch(id=5, input_ops=[], num_prefetch_pages=256), MatMulOp(id=6, input_ops=[SramPrefetch(id=5, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16), SramPrefetchRelease(id=7, input_ops=[MatMulOp(id=6, input_ops=[SramPrefetch(id=5, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)]), SramPrefetch(id=8, input_ops=[], num_prefetch_pages=256), MatMulOp(id=9, input_ops=[SramPrefetch(id=8, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16), SramPrefetchRelease(id=10, input_ops=[MatMulOp(id=9, inp

In [8]:
for inst in macro_op_list:
    print(inst)

VectorOp(id=1, input_ops=[], vector_op_type='rms_norm', vector_shape=[2, 4096])
SramPrefetch(id=2, input_ops=[], num_prefetch_pages=256)
MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)
SramPrefetchRelease(id=4, input_ops=[MatMulOp(id=3, input_ops=[SramPrefetch(id=2, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)])
SramPrefetch(id=5, input_ops=[], num_prefetch_pages=256)
MatMulOp(id=6, input_ops=[SramPrefetch(id=5, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)
SramPrefetchRelease(id=7, input_ops=[MatMulOp(id=6, input_ops=[SramPrefetch(id=5, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)])
SramPrefetch(id=8, input_ops=[], num_prefetch_pages=256)
MatMulOp(id=9, input_ops=[SramPrefetch(id=8, input_ops=[], num_prefetch_pages=256)], dim=(2, 4096, 128), weight_bits=16)
SramPrefetchRelease(id=10, input_ops=[MatMulOp(id=9, input_ops=[Sr

In [9]:
len(macro_op_list)

1187

测试完整的集成工作

直接复用前面 cell 里已经生成好的 `nand_config`、`inference_config`、`kv_cache_state` 和 `macro_op_list`：
- 初始化 xPU 模拟器
- 正确加载 frontend 产出的各种 commands
- 异步运行并输出最终时间

In [10]:
from nandmachine.simulator.entry_point import run_macro_ops

simulator_config = {
    "device_name": "A100_80GB",
    "compile_mode": "heuristic-GPU",
    "weight_bits": 16,
}
final_cycle = run_macro_ops(
    nand_config,
    macro_op_list,
    **simulator_config,
)

assert final_cycle > 0
print("macro op count:", len(macro_op_list))
print("macro op type summary:", dict(macro_summary(macro_op_list)))
print("nand config:", nand_config)
print("inference config:", inference_config)
print("kv cache state:", kv_cache_state)
print("simulator config:", simulator_config)
print("final sim cycle:", final_cycle)


macro op count: 1187
macro op type summary: {'VectorOp': 5, 'SramPrefetch': 394, 'MatMulOp': 370, 'SramPrefetchRelease': 394, 'FlashAttnOp': 24}
nand config: NandConfig(num_channels=1, num_plane=1, num_block=4, num_pages=16, tRead=1.0, tWrite=2.0, tErase=3.0, page_size=4, sram_threshold=1024)
inference config: InferenceConfig(batch_size=2, input_sequence_length=8, output_sequence_length=4, weight_bits=16, activation_bits=16, kv_cache_bits=16, kv_block_size_bytes=1024, parallel_config=ParallelConfig(num_ranks=1))
kv cache state: KVCacheState(total_kv_cache_size_per_layer=98304, num_nand_pages_per_layer=24, num_hyper_pages_per_layer=24, kv_block_size_tokens=1, num_kv_blocks=96, kv_cache_num_pages_per_layer=24)
simulator config: {'device_name': 'A100_80GB', 'compile_mode': 'heuristic-GPU', 'weight_bits': 16}
final sim cycle: 309884
